# Cell-line join — does it pair the right two lines?

Analysis, not pipeline. Checks the join the whole project rests on: SCP542's cell-line names against
CTRPv2's, which is how expression is matched to response.

| | |
|---|---|
| **reads** | `..._with_targets_auc_cc.h5ad` — `obs` only, for the resolved lines |
| | `drevalpy_.../CTRPv2/CTRPv2.csv`, `cell_line_names.csv` — their identifiers |
| **writes** | nothing — it prints, and is read rather than consumed |

**Why the join is on names and not accessions.** Both are available. Cellosaurus accessions are the
principled choice and they lose lines: names recover 181 overlapping cell lines where the accessions
DrEval ship recover 172. The decision is made on that evidence rather than on the principle — and the
opposite decision was made for compounds, where `master_cpd_id` is exact and names lose 102 of 545.

**Why it needs checking at all.** A name join can pair two *different* lines that happen to normalise
to the same string, and nothing downstream would notice: the model would simply learn from a response
measured on another cell line.

In [1]:
from pathlib import Path

import anndata as ad
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts').is_dir())
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.layout import DEFAULT_CTRP_SCORE, PipelinePaths

# Derived, not hardcoded (13.08.2026, #25): both were absolute /Users paths. DREVAL resolves
# to the identical directory -- the record number comes from layout.ZENODO_RESPONSE_RECORD,
# so it can no longer drift from the pinned record.
#
# ** TARGETS CHANGES WHICH FILE IS READ, and that is Selin's call, not a tidy-up. ** The
# literal ended in _auc.h5ad -- the target RETIRED on 11.08.2026 as defectively normalised.
# Deriving resolves to the current default, 'auc_cc'. Spelled out rather than left implicit.
_paths = PipelinePaths.build(None, 'hvg5000', DEFAULT_CTRP_SCORE)
DREVAL = _paths.drevalpy_dir
TARGETS = _paths.targets_h5ad
print(f'target artifact: {TARGETS.name}')

# The production join key, verbatim from ctrp_to_h5ad._normalize_cell_line. Anything looser would be
# testing a different join than the one that runs.
def prod_key(s):
    return str(s).strip().lower().replace('-', '')


# Cellosaurus writes names with spaces and dots that our key never has to handle, because it only ever
# sees SCP542's and CTRP's spellings. Used ONLY to look names up in Cellosaurus, never as a join key.
def cello_key(s):
    return prod_key(s).replace(' ', '').replace('.', '')


# SCP542 labels are like MDAMB361_BREAST: the line, then the tissue. Both halves are used -- the name
# for the join, the tissue as an independent check that does not go through Cellosaurus at all.
obs = ad.read_h5ad(TARGETS, backed='r').obs
scp = (pd.DataFrame({'scp_label': obs['Cell_line'].astype(str).unique()})
       .assign(scp_name=lambda d: d.scp_label.str.split('_').str[0],
               scp_tissue=lambda d: d.scp_label.str.split('_', n=1).str[1].fillna(''))
       .assign(key=lambda d: d.scp_name.map(prod_key)))

# One row per CTRP cell line: its name, the accession DrEval ships for it, and its tissue.
ctrp = (pd.read_csv(DREVAL / 'CTRPv2' / 'CTRPv2.csv', usecols=['cellosaurus_id', 'ccl_name'])
        .drop_duplicates('ccl_name')
        .merge(pd.read_csv(DREVAL / 'CTRPv2' / 'cell_line_names.csv'), on='cellosaurus_id', how='left')
        .assign(key=lambda d: d.ccl_name.map(prod_key)))

print(f'SCP542 cell lines : {len(scp):,}')
print(f'CTRPv2 cell lines : {len(ctrp):,}')
print(f'distinct SCP542 names collapsing onto one key : '
      f'{int((scp.groupby("key").scp_name.nunique() > 1).sum())}')
print(f'distinct CTRP names collapsing onto one key   : '
      f'{int((ctrp.groupby("key").ccl_name.nunique() > 1).sum())}')

target artifact: SCP542_CCLE_scGPT_human_embeddings_with_targets_auc_cc.h5ad


SCP542 cell lines : 198
CTRPv2 cell lines : 886
distinct SCP542 names collapsing onto one key : 0
distinct CTRP names collapsing onto one key   : 0


## 1 · Resolving both sides to an accession

| | |
|---|---|
| **out** | each side's names mapped to Cellosaurus accessions, where one exists |

Accessions are not used to *make* the join — they are used to audit it. Resolving both sides
independently gives a second opinion on every pair the names produced.

In [2]:
import sys

sys.path.insert(0, str(ROOT))
from scripts.sources.cellosaurus import load_cellosaurus, resolve_accessions  # noqa: E402

cello, release = load_cellosaurus(DREVAL / 'meta' / 'cellosaurus.txt')
print(f'{release}: {cello.accession.nunique():,} entries, {len(cello):,} names '
      f'({int((cello.kind == "synonym").sum()):,} of them synonyms)')

resolved = resolve_accessions(scp.scp_name, cello)
print('\nstatus of the 198 SCP542 names:')
print(resolved.status.value_counts().to_string())

# Every name that needed a tie-break, and which rule settled it -- the table in the header above.
print('\nnames matching more than one entry:')
print(resolved[resolved.n_candidates > 1]
      [['name', 'n_candidates', 'accession', 'kind', 'resolved_by', 'status']]
      .sort_values(['resolved_by', 'name']).to_string(index=False))

print('\nnames absent from Cellosaurus entirely:',
      resolved.loc[resolved.status == 'absent', 'name'].tolist())

Cellosaurus 52.0 (10-April-2025): 163,868 entries, 288,394 names (124,526 of them synonyms)



status of the 198 SCP542 names:
status
resolved    196
absent        2

names matching more than one entry:
 name  n_candidates accession       kind                          resolved_by   status
 EBC1             2 CVCL_2891 identifier a primary identifier beats a synonym resolved
  PC3             7 CVCL_0035 identifier a primary identifier beats a synonym resolved
SCC25             2 CVCL_1682 identifier a primary identifier beats a synonym resolved
 SCC9             2 CVCL_1685 identifier a primary identifier beats a synonym resolved
  C32             3 CVCL_1097    synonym               cancer cell lines only resolved
 RCM1             2 CVCL_1648    synonym               cancer cell lines only resolved
 A204             2 CVCL_1058 identifier                           human only resolved
 ABC1             2 CVCL_1066 identifier                           human only resolved
 HT55             2 CVCL_1294 identifier                           human only resolved
  TE1             2 C

## 2 · Does the join pair the right two lines?

| | |
|---|---|
| **out** | agreement between the name pairing and the accession pairing, plus a tissue cross-check |

Two independent checks. Where both sides resolve to an accession, the accessions must agree. Where
they do not resolve, the CCLE primary site is compared against CTRPv2's recorded tissue — a weaker
check, but it catches a pairing of two unrelated lines.

**Why the tissue check is kept despite being weak.** It is the only check available for the lines
that have no accession on one side, which is exactly the population where a name collision is most
likely to go unnoticed.

In [3]:
joined = (scp.merge(ctrp, on='key', how='inner')
          .merge(resolved[['name', 'accession', 'status']],
                 left_on='scp_name', right_on='name', how='left'))
print(f'SCP542 lines matched to a CTRP line by name: {len(joined)} of {len(scp)}')

agree = joined.accession == joined.cellosaurus_id
unresolvable = joined.accession.isna()
print(f'  accessions agree      : {int((agree & ~unresolvable).sum())}')
print(f'  accessions DISAGREE   : {int((~agree & ~unresolvable).sum())}')
print(f'  SCP542 name unresolved: {int(unresolvable.sum())}')
if int((~agree & ~unresolvable).sum()):
    print(joined[~agree & ~unresolvable]
          [['scp_name', 'accession', 'ccl_name', 'cellosaurus_id', 'tissue']].to_string(index=False))

# --- independent of Cellosaurus: SCP542's tissue suffix vs DrEval's tissue ---
# The two vocabularies differ; these pairs are the same tissue under different names, checked by hand
# against the lines they affect (bladder lines filed under urinary tract, cholangiocarcinoma lines
# under liver, rhabdomyosarcoma lines under muscle).
TISSUE_SYNONYMS = {
    'URINARYTRACT': 'BLADDER', 'BILIARYTRACT': 'LIVER', 'SOFTTISSUE': 'MUSCLE',
    'LARGEINTESTINE': 'COLON', 'AUTONOMICGANGLIA': 'NERVOUSSYSTEM', 'PLEURA': 'LUNG',
    'HAEMATOPOIETICANDLYMPHOID': 'BLOOD', 'UPPERAERODIGESTIVETRACT': 'HEADANDNECK',
    'CENTRALNERVOUSSYSTEM': 'BRAIN', 'OESOPHAGUS': 'ESOPHAGUS', 'ENDOMETRIUM': 'UTERUS',
    'BONE': 'SARCOMA', 'BILIARYTRACT ': 'BILEDUCT',
}


def tissue_agrees(scp_tissue, dreval_tissue):
    a = str(scp_tissue).upper().replace('_', '').replace(' ', '')
    b = str(dreval_tissue).upper().replace('_', '').replace(' ', '')
    return a == b or a in b or b in a or TISSUE_SYNONYMS.get(a) == b


joined['tissue_ok'] = [tissue_agrees(a, b) for a, b in zip(joined.scp_tissue, joined.tissue)]
print(f'\ntissue check over the same {len(joined)} matches:')
print(f'  agree                 : {int(joined.tissue_ok.sum())}')
print(f'  disagree              : {int((~joined.tissue_ok).sum())}')
if int((~joined.tissue_ok).sum()):
    print(joined[~joined.tissue_ok][['scp_name', 'scp_tissue', 'ccl_name', 'tissue']]
          .to_string(index=False))

SCP542 lines matched to a CTRP line by name: 180 of 198
  accessions agree      : 180
  accessions DISAGREE   : 0
  SCP542 name unresolved: 0

tissue check over the same 180 matches:
  agree                 : 180
  disagree              : 0


## 3 · What the target build actually produces

| | |
|---|---|
| **out** | the line counts the pipeline ends up with, compared against what the join predicts |

The audit is only useful if it describes the join the pipeline performs, not an idealised one. This
section reads the built targets file and confirms the overlap it contains matches what §2 verified.

Two findings are recorded here rather than in prose elsewhere: CTRPv2 spells one line differently, so
the name join silently dropped a *screened* line until an explicit sourced alias was added; and an
experiment listed once per calendar day it ran was double-counted in the per-(line, drug) mean.

In [4]:
import numpy as np  # noqa: E402

from scripts.preprocessing import ctrp_to_h5ad as C  # noqa: E402
from scripts.layout import CTRP_SCORES  # noqa: E402

# The pipeline's own key function, applied to the same names -- vectorised over the Series, which is
# how ctrp_to_h5ad calls it.
scp_keys = set(C._normalize_cell_line(scp.scp_name))
summary = []
for score in CTRP_SCORES:
    print(f'===== {score} ({C.SCORE_COLUMNS[score]}) =====')
    full = C._deduplicate_measurements(C._load_drevalpy_long(DREVAL / 'CTRPv2' / 'CTRPv2.csv', score))
    overlap = scp_keys & set(full.ccl_name_norm)
    long_ov, kept = C._build_drug_table(full, overlap_cell_lines_norm=overlap,
                                        min_cell_lines=50, target_drugs=None)
    Y = long_ov.pivot(index='ccl_name_norm', columns='cpd_name_norm',
                      values='score').reindex(columns=kept)
    v = Y.to_numpy()[~np.isnan(Y.to_numpy())]
    print(f'  overlap {len(overlap)} lines | matrix {Y.shape} | observed {int(Y.notna().sum().sum()):,} '
          f'| density {100 * Y.notna().mean().mean():.1f} %')
    print(f'  values: min {v.min():.3f}  median {np.median(v):.3f}  max {v.max():.3f}\n')
    summary.append({'measure': score, 'lines': Y.shape[0], 'drugs': Y.shape[1],
                    'observed': int(Y.notna().sum().sum()),
                    'density_%': round(100 * Y.notna().mean().mean(), 1),
                    'min': round(float(v.min()), 3), 'median': round(float(np.median(v)), 3),
                    'max': round(float(v.max()), 3)})

print(pd.DataFrame(summary).to_string(index=False))

===== auc_cc (AUC_curvecurator) =====


  395,024 measurements | 886 cell lines | 545 drugs
  8,187 of 395,024 rows are exact duplicates of another row (2.1 %) and are dropped.
  Drug filter: 534 / 545 drugs kept (>= 50 overlapping cell lines).
  overlap 181 lines | matrix (181, 534) | observed 81,906 | density 84.7 %
  values: min 0.020  median 0.925  max 1.830

===== ln_ic50_cc (LN_IC50_curvecurator) =====


  159,050 of 395,024 rows have no LN_IC50_curvecurator (40.3 %) and are dropped.
  235,974 measurements | 886 cell lines | 545 drugs
  4,346 of 235,974 rows are exact duplicates of another row (1.8 %) and are dropped.
  Drug filter: 365 / 539 drugs kept (>= 50 overlapping cell lines).
  overlap 181 lines | matrix (181, 365) | observed 42,584 | density 64.5 %
  values: min -11.385  median 2.487  max 8.634

   measure  lines  drugs  observed  density_%     min  median   max
    auc_cc    181    534     81906       84.7   0.020   0.925 1.830
ln_ic50_cc    181    365     42584       64.5 -11.385   2.487 8.634
